# InkSight

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# ---------- Read csv ----------
error_data = pd.read_csv("error_data_clean.csv")
filterflow_data = pd.read_csv("filterflow_data.csv")

filterflow_data['timestamp'] = pd.to_datetime(filterflow_data['timestamp'])
error_data['timestamp'] = pd.to_datetime(error_data['timestamp'])

# ---------- add is_error label（±7 hours）----------
filterflow_data['is_error'] = 0
for idx, row in error_data.iterrows():
    p, c, t = row['printer'], row['color'], row['timestamp']
    mask = (
        (filterflow_data['printer'] == p) &
        (filterflow_data['color'] == c) &
        (abs((filterflow_data['timestamp'] - t).dt.total_seconds()) <= 25200)
    )
    filterflow_data.loc[mask, 'is_error'] = 1

# ---------- Features engineering ----------
def extract_features(df, use_error_features=False):
    df = df.copy()
    df['weekday'] = df['timestamp'].dt.weekday
    df['hour'] = df['timestamp'].dt.hour
    df['day_night'] = df['hour'].apply(lambda x: 'day' if 8 <= x <= 20 else 'night')

    stats = df.groupby(['printer', 'color'])['filter_flow'].agg(['mean', 'max', 'min']).reset_index()
    stats.columns = ['printer', 'color', 'avg_flow', 'max_flow', 'min_flow']
    df = df.merge(stats, on=['printer', 'color'], how='left')

    # not using features about is_error
    df['error_count'] = 0
    df['error_flow_avg'] = 0
    df['error_flow_min'] = 0
    df['error_flow_max'] = 0

    return df

# ---------- Model training ----------
filterflow_data = extract_features(filterflow_data, use_error_features=False)
filterflow_data = pd.get_dummies(filterflow_data, columns=['day_night'], drop_first=True)

features = ['filter_flow', 'weekday', 'avg_flow', 'max_flow', 'min_flow', 'day_night_night']

df_model = filterflow_data.dropna(subset=features + ['is_error'])
X = df_model[features]
y = df_model['is_error']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ---------- Evaluate model ----------
y_pred = model.predict(X_test)
if len(model.classes_) == 2:
    y_prob = model.predict_proba(X_test)[:, 1]
    print(classification_report(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, y_prob))
else:
    print("Can't give probability")

# ---------- Predict new data ----------
df_new = pd.read_csv("test.csv")
df_new['timestamp'] = pd.to_datetime(df_new['timestamp'])

# Keep original for output
df_meta = df_new[['timestamp', 'printer', 'color', 'filter_flow']].copy()

df_new = extract_features(df_new, use_error_features=False)
df_new = pd.get_dummies(df_new, columns=['day_night'], drop_first=True)

# Fill in any missing columns
for col in features:
    if col not in df_new.columns:
        df_new[col] = 0

df_new = df_new[features]

# Predict
if len(model.classes_) == 2:
    df_meta['error_probability'] = model.predict_proba(df_new)[:, 1]
else:
    df_meta['error_probability'] = 0.0

# ---------- Print results ----------
df_sorted = df_meta.sort_values(by='error_probability', ascending=False)
print(df_sorted[['timestamp', 'printer', 'color', 'filter_flow', 'error_probability']])

# Optional: Save results
# df_sorted.to_csv("prediction_results.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importances = model.feature_importances_
sns.barplot(x=importances, y=X.columns)
plt.title("Feature Importance in Random Forest")
plt.show()
